In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
import torch.distributions as TD
from tqdm import tqdm

import wandb
from src.costs.lse import MLPLSECost
from src.costs.mlp_based import MLPCost, MLPL2Cost
from src.models.energy_based import EGEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import plot_B_parameters
from src.potentials.mlp_based import MLPPotential
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.energy_based.cost import MLPCostConfig, MLPLSECostConfig, MLPL2CostConfig
from configs.energy_based.dataset import DatasetConfig, MiniBatchConfig
from configs.energy_based.model import EBMConfig
from configs.energy_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.energy_based.potential import PotentialConfig
from configs.energy_based.sampling import LangevinConfig
from configs.energy_based.train import TrainConfig

In [6]:
Q_X_UNPAIRED_SAMPLES = 6990
R_Y_UNPAIRED_SAMPLES = 7141
P_XY_PAIRED_SAMPLES = 128
LR_PAIRED = 5e-4
LR_UNPAIRED = 2e-4
SAMPLING_NUM_ITER = 100
MAX_STEPS = 1000
COST_FUNCTION = "MLP"
PAIRED_BATCH_SIZE = 128
UNPAIRED_BATCH_SIZE = 128

In [7]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
# minibatch_config = MiniBatchConfig()

# potential_config = PotentialConfig(hidden_layers=POTENTIAL_HIDDEN_LAYERS)
# if COST_FUNCTION == "MLP":
#     cost_config = MLPCostConfig(hidden_layers=HIDDEN_LAYERS)
#     EXP_META_INFO = f"HIDDEN_LAYERS_{HIDDEN_LAYERS}_"
# elif COST_FUNCTION == "MLPLSE":
#     cost_config = MLPLSECostConfig(
#         m_potentials=M_POTENTIALS,
#         log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
#         b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
#     )
#     EXP_META_INFO = (
#         f"M_POTENTIALS_{M_POTENTIALS}_"
#         + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
#         + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
#     )
# elif COST_FUNCTION == "MLPL2":
#     cost_config = MLPL2CostConfig(
#         x_hidden_layers=X_HIDDEN_LAYERS,
#         y_hidden_layers=Y_HIDDEN_LAYERS,
#     )
#     EXP_META_INFO = f"X_HIDDEN_LAYERS_{X_HIDDEN_LAYERS}_" + f"Y_HIDDEN_LAYERS_{Y_HIDDEN_LAYERS}_"
# else:
#     raise ValueError(f"Unknown cost function: {COST_FUNCTION}!")
model_config = EBMConfig(sampling=LangevinConfig(num_iterations=SAMPLING_NUM_ITER))

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [8]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [9]:
# from src.utils.color_mnist import download_colored_mnist_data
from torchvision.transforms import Compose, Resize, Normalize, ToTensor, Lambda
import torchvision.datasets as datasets

In [17]:
DATASET = 'mnist2usps'
IMG_SIZE = 32
DATASET_PATH = '../datasets/'

In [18]:
source_subset = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7 ,8, 9])
new_labels_source = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7, 8:8, 9:9}
target_subset = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
new_labels_target = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7, 8:8, 9:9}
scheduler_milestones=[10000, 20000, 30000, 40000, 50000]

source_transform = Compose([
    Resize((IMG_SIZE, IMG_SIZE)), 
    ToTensor(),
    Normalize((0.5), (0.5)),
])
target_transform = source_transform

if DATASET == 'mnist2kmnist':
    source = datasets.MNIST
    target = datasets.KMNIST
    
elif DATASET == 'fmnist2mnist':
    source = datasets.FashionMNIST
    target = datasets.MNIST
    
elif DATASET == 'mnist2usps':
    source = datasets.MNIST
    target = datasets.USPS
    
elif DATASET == 'mnist2mnistm':
    source = datasets.MNIST
    target = MNISTM
    NC = 3
    source_transform = Compose([
        Resize((IMG_SIZE, IMG_SIZE)), 
        ToTensor(),
        Normalize((0.5), (0.5)), 
        Lambda(lambda x: -x.repeat(3,1,1))])
    target_transform = Compose([
        Resize(IMG_SIZE),
        ToTensor(),
        Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    
OUTPUT_PATH = '../saved_models/{}/'.format(DATASET)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

In [19]:
source_train = source(root=DATASET_PATH, train=True, download=True, transform=source_transform)
target_train = target(root=DATASET_PATH, train=True, download=True, transform=target_transform) 

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 6579383/6579383 [00:09<00:00, 661448.04it/s]


In [20]:
# src_data_name = "MNISTcolored_2"
# trg_data_name = "MNISTcolored_3"
# download_colored_mnist_data(src_data_name)
# download_colored_mnist_data(trg_data_name)
src_digit = 2
trgt_digit = 2

In [45]:
import torchvision.transforms as tr
import torchvision.datasets as datasets

In [46]:
IMAGE_SIZE = 32
IMAGE_CHANNELS = 3

In [47]:
transform = tr.Compose(
    [
        tr.Resize(IMAGE_SIZE),
        tr.CenterCrop(IMAGE_SIZE),
        tr.ToTensor(),
        tr.Normalize(tuple(0.5 * torch.ones(IMAGE_CHANNELS)), tuple(0.5 * torch.ones(IMAGE_CHANNELS))),
    ]
)

In [48]:
q_x = torch.stack([x[0] for x in datasets.ImageFolder(root=f"./datasets/MNIST/{src_digit}", transform=transform)]).to(device)
q_y = torch.stack([x[0] for x in datasets.ImageFolder(root=f"./datasets/MNIST/{trgt_digit}", transform=transform)]).to(device)

In [49]:
from src.samplers.base import TensorSampler
from src.utils.paired import get_paired_sampler

In [50]:
X_sampler = TensorSampler(q_x.to(dtype), device=device)
Y_sampler = TensorSampler(q_y.to(dtype), device=device)

In [51]:
X_paired_train = q_x[:P_XY_PAIRED_SAMPLES].clone()
Y_paired_train = q_y[:P_XY_PAIRED_SAMPLES].clone()

In [52]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [53]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [54]:
from src.costs.convolutional import NonlocalCost, VanillaCost
from src.potentials.convolutional import NonlocalPotential, VanillaPotential

In [55]:
potential = VanillaPotential(n_f=IMAGE_SIZE) # NonlocalPotential(n_f=IMAGE_SIZE)

In [56]:
cost = VanillaCost(n_f=IMAGE_SIZE)

In [57]:
# TODO: add to config
BASIC_NOISE_VAR = 1.0
P_SAMPLE_BUFFER_REPLAY = 0.95
SAMPLE_BUFFER_SAMPLES = 10000

In [58]:
basic_noise_gen = TD.Normal(torch.zeros_like(q_x[0]).to(device), torch.ones_like(q_x[0]).to(device) * BASIC_NOISE_VAR)

sample_buffer_instance = SampleBufferEgEOT(
    basic_noise_gen, p=P_SAMPLE_BUFFER_REPLAY, max_samples=SAMPLE_BUFFER_SAMPLES, device=device
)

In [59]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [60]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [61]:
D_opt_unpaired = torch.optim.Adam(model.potential.parameters(), **opt_unpaired_config.model_dump())

In [62]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [63]:
# TODO: refactor this config
EXP_NAME = (
    "EgEOT_ColoredMNIST_"
    + f"COST_FUNCTION_{COST_FUNCTION}_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"SAMPLING_STEPS_{model_config.sampling.num_iterations}_"
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    COST_FUNCTION=COST_FUNCTION,
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [64]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [65]:
# starting_points = torch.tensor([[-2.0, 0.0], [0.0, 0.0], [0.0, -2.0]])
# num_ending_points = 64

In [66]:
# num_starting_points_paired = 5
# indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
# starting_points_paired = X_paired_train[indices]
# ending_points_paired = Y_paired_train[indices]

In [67]:
# gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

In [68]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y, compute_stats=True)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired: Loss": D_loss_unpaired.item()}, step=step)
    wandb.log({f"Unpaired: \int f(y)": output_unpaired["int_potential"].item()}, step=step)
    wandb.log({f"Unpaired: \int\log Z": output_unpaired["int_log_Z"].item()}, step=step)
    wandb.log({f"Unpaired: -E(x, y)": output_unpaired["neg_energy_t"].item()}, step=step)
    wandb.log({f"Unpaired: c(x, y)": output_unpaired["cost_t"].item()}, step=step)
    wandb.log({f"Unpaired: f(y)": output_unpaired["potential_t"].item()}, step=step)
    wandb.log({f"Unpaired: noise": output_unpaired["noise"].item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    output_paired = model.compute_paired_loss(X_paired, Y_paired, compute_stats=True)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired: Loss": D_loss_paired.item()}, step=step)
    # wandb.log({f"Paired: -E(x, y)": output_paired["neg_energy_t"].item()}, step=step)
    # wandb.log({f"Paired: c(x, y)": output_paired["cost_t"].item()}, step=step)
    # wandb.log({f"Paired: f(y)": output_paired["potential_t"].item()}, step=step)
    # wandb.log({f"Paired: noise": output_paired["noise"].item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    # wandb.log(
    #     {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )
    # wandb.log(
    #     {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )

    if step % train_config.plot_every == 0:
        # distr_dict = plot_swiss_roll(
        #     {"EBM": model},
        #     X_sampler,
        #     Y_sampler,
        #     X_paired_train,
        #     Y_paired_train,
        #     starting_points,
        #     gt_Y_points,
        #     log=True,
        # )
        # if COST_FUNCTION == "MLPLSE":
        #     B_dict = B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        #     distr_dict = distr_dict | B_dict
        # wandb.log(distr_dict)
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"model_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

 59%|█████████████████████████████████████████████████████████████████████████▊                                                    | 586/1000 [32:22<22:52,  3.32s/it]


KeyboardInterrupt: 

## Plotting

In [ ]:
plot_swiss_roll(
    {"EBM": model},
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 